# Matplotlib Example - SL Assignment 2026

This notebook contains two tasks:

1. Generate 1000 random sine/cosine sequences and use an LSTM with 100 hidden units to predict the tested wave.
2. Analyze a large text file sentiment using TextBlob and show the result with Matplotlib.

## Install Required Libraries

Run this cell once if the libraries are not installed.

In [5]:
%pip install numpy matplotlib tensorflow textblob


^C
Note: you may need to restart the kernel to use updated packages.


## Task 1: LSTM Prediction For Random Sine/Cosine Waves

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from textblob import TextBlob


ModuleNotFoundError: No module named 'matplotlib'

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\adelm\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\matplotlib\\backends\\_backend_agg.cp313-win_amd64.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 3.9 MB/s eta 0:00:02
   ----- ---------------------------------- 1.0/8.2 MB 2.8 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.2 MB 2.9 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.2 MB 3.2 MB/s eta 0:00:02
   -------------- ------------------------- 2.9/8.2 MB 3.2 MB/s eta 0:00:02
   ---------------- ----------------------- 3.4/8.2 MB 2.9 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.2 MB 2.7 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.2 MB 2.7 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 2.4 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.2 MB 2.3 MB/s

In [ ]:
def create_random_wave_sequences(number_of_sequences=1000, sequence_length=80, random_seed=42):
    rng = np.random.default_rng(random_seed)
    time_steps = np.linspace(0, 2 * np.pi, sequence_length)
    waves = []

    for _ in range(number_of_sequences):
        wave_type = rng.choice(["sine", "cosine"])
        amplitude = rng.uniform(0.5, 2.0)
        frequency = rng.uniform(0.5, 3.0)
        phase = rng.uniform(0, 2 * np.pi)
        noise = rng.normal(0, 0.03, size=sequence_length)

        if wave_type == "sine":
            wave = amplitude * np.sin(frequency * time_steps + phase)
        else:
            wave = amplitude * np.cos(frequency * time_steps + phase)

        waves.append(wave + noise)

    waves = np.array(waves, dtype=np.float32)
    X = waves[:, :-1].reshape(number_of_sequences, sequence_length - 1, 1)
    y = waves[:, -1].reshape(number_of_sequences, 1)
    return X, y, waves


def build_lstm_model(input_steps):
    model = Sequential([
        LSTM(100, input_shape=(input_steps, 1)),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss="mse")
    return model

In [ ]:
number_of_sequences = 1000
sequence_length = 80
max_epochs = 10

X, y, waves = create_random_wave_sequences(number_of_sequences, sequence_length)

train_size = int(0.8 * number_of_sequences)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

model = build_lstm_model(sequence_length - 1)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=max_epochs,
    batch_size=32,
    verbose=1
)

test_mse = model.evaluate(X_test, y_test, verbose=0)
predictions = model.predict(X_test, verbose=0)

print(f"Final training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")
print(f"Test MSE: {test_mse:.6f}")
print(f"Actual tested wave next value: {y_test[0, 0]:.6f}")
print(f"Predicted tested wave next value: {predictions[0, 0]:.6f}")

In [ ]:
tested_wave_index = train_size
known_points = waves[tested_wave_index, :-1]
actual_next = y_test[0, 0]
predicted_next = predictions[0, 0]

plt.figure(figsize=(10, 5))
plt.plot(range(len(known_points)), known_points, label="Known tested wave")
plt.scatter(len(known_points), actual_next, label="Actual next value", color="green", s=80)
plt.scatter(len(known_points), predicted_next, label="Predicted next value", color="red", s=80)
plt.title("LSTM Prediction For Tested Sine/Cosine Wave")
plt.xlabel("Time step")
plt.ylabel("Wave value")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("LSTM Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Task 2: Sentiment Analysis For A Large Text File With TextBlob

In [ ]:
from pathlib import Path
from textblob import TextBlob

In [ ]:
def classify_sentiment(polarity, threshold=0.05):
    if polarity > threshold:
        return "positive"
    if polarity < -threshold:
        return "negative"
    return "neutral"


def iter_text_chunks(file_path, chunk_size=5000):
    with Path(file_path).open("r", encoding="utf-8", errors="ignore") as file:
        buffer = []
        current_size = 0

        for line in file:
            buffer.append(line)
            current_size += len(line)

            if current_size >= chunk_size:
                yield "".join(buffer)
                buffer = []
                current_size = 0

        if buffer:
            yield "".join(buffer)


def analyze_large_text_file(file_path, chunk_size=5000):
    total_weight = 0
    weighted_polarity = 0.0
    weighted_subjectivity = 0.0
    sentiment_counts = {"positive": 0, "neutral": 0, "negative": 0}
    chunk_count = 0

    for chunk in iter_text_chunks(file_path, chunk_size):
        clean_chunk = chunk.strip()
        if not clean_chunk:
            continue

        blob = TextBlob(clean_chunk)
        polarity = blob.sentiment.polarity
        subjectivity = blob.sentiment.subjectivity
        weight = len(clean_chunk)

        weighted_polarity += polarity * weight
        weighted_subjectivity += subjectivity * weight
        total_weight += weight
        chunk_count += 1
        sentiment_counts[classify_sentiment(polarity)] += 1

    if total_weight == 0:
        raise ValueError("The input file does not contain analyzable text.")

    overall_polarity = weighted_polarity / total_weight
    overall_subjectivity = weighted_subjectivity / total_weight

    return {
        "chunks_analyzed": chunk_count,
        "positive_chunks": sentiment_counts["positive"],
        "neutral_chunks": sentiment_counts["neutral"],
        "negative_chunks": sentiment_counts["negative"],
        "polarity": overall_polarity,
        "subjectivity": overall_subjectivity,
        "sentiment": classify_sentiment(overall_polarity),
    }

Set `file_path` to your large text file. The next cell creates a small sample file only so the notebook can run immediately.

In [ ]:
sample_text = """
I love this course. The examples are useful and the explanation is clear.
Sometimes the task is difficult, but practice makes it easier.
The final result is good and I am happy with the progress.
""" * 300

sample_file = Path("sample_large_text.txt")
sample_file.write_text(sample_text, encoding="utf-8")

# Change this path to your real large text file when needed.
file_path = sample_file

In [ ]:
results = analyze_large_text_file(file_path, chunk_size=5000)

print("TextBlob Sentiment Analysis")
print("---------------------------")
print(f"File: {file_path}")
print(f"Chunks analyzed: {results['chunks_analyzed']}")
print(f"Positive chunks: {results['positive_chunks']}")
print(f"Neutral chunks: {results['neutral_chunks']}")
print(f"Negative chunks: {results['negative_chunks']}")
print(f"Overall polarity: {results['polarity']:.4f}")
print(f"Overall subjectivity: {results['subjectivity']:.4f}")
print(f"Overall sentiment: {results['sentiment']}")

In [ ]:
labels = ["positive", "neutral", "negative"]
values = [
    results["positive_chunks"],
    results["neutral_chunks"],
    results["negative_chunks"],
]
colors = ["#2e7d32", "#607d8b", "#c62828"]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=colors)
plt.title("TextBlob Sentiment Analysis")
plt.xlabel("Sentiment")
plt.ylabel("Number of chunks")

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height, str(int(height)), ha="center", va="bottom")

summary = f"Overall sentiment: {results['sentiment']} | Polarity: {results['polarity']:.4f}"
plt.figtext(0.5, 0.01, summary, ha="center", fontsize=10)
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()